# Agent Gauntlet: GRPO + Unsloth (Judge-Ready, End-to-End)

This notebook is built for hackathon judging with **real environment-connected training** and **real evidence artifacts**.

## What this notebook proves

- GRPO training runs against the live Agent Gauntlet environment interface (not static dataset rewards).
- A trained checkpoint is evaluated against a random baseline with matched seeds.
- Evidence artifacts are saved to repo paths for README:
  - `assets/reward_curves.png`
  - `assets/component_rewards.png`
  - `assets/trained_vs_random.png`
  - `assets/trained_vs_random.json`

> If you run in Colab, keep runtime on GPU (T4/A10/A100).

In [ ]:
# Cell 1: Setup (local or Colab, one-click rerunnable)
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Self-contained install path for fresh Colab runtimes.
    get_ipython().run_line_magic('pip', 'install -U pip')
    get_ipython().run_line_magic('pip', 'install -e .[dev]')
    get_ipython().run_line_magic('pip', 'install -U unsloth "trl[vllm]" wandb matplotlib')

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Workspace:', ROOT)
print('Python:', sys.version.split()[0])
print('In Colab:', IN_COLAB)

In [ ]:
# Cell 2: Config (judge-grade defaults)
from datetime import datetime

DIFFICULTY = 'easy'          # easy | medium | hard | expert
DATASET_SIZE = 1600          # ensures meaningful update count with grad_accum=32
NUM_EPOCHS = 2               # medium-long run for credible learning curves
EPISODES_EVAL = 30           # stronger evaluation confidence than tiny runs
MODEL_ID = 'Qwen/Qwen3-1.7B'
USE_WANDB = False            # set True if you want online run tracking
WANDB_PROJECT = 'agent-gauntlet'
JUDGE_READY = True           # enforce minimum training budget in train_grpo.py
ENV_URL = 'http://127.0.0.1:8000'

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = f'outputs/gauntlet-{DIFFICULTY}-{RUN_TAG}'
print('Output dir:', OUTPUT_DIR)
print('ENV_URL:', ENV_URL)

In [ ]:
# Cell 3: Start local environment server (required for Colab reruns)
import os
import subprocess
import sys
import time
import urllib.request

server_cmd = [
    sys.executable,
    '-m',
    'uvicorn',
    'server.app:app',
    '--host',
    '127.0.0.1',
    '--port',
    '8000',
]

server_proc = subprocess.Popen(server_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print('Server PID:', server_proc.pid)

deadline = time.time() + 60
healthy = False
while time.time() < deadline:
    try:
        with urllib.request.urlopen(f'{ENV_URL}/health', timeout=2) as r:
            if r.status == 200:
                healthy = True
                break
    except Exception:
        time.sleep(1)

if not healthy:
    if server_proc.poll() is not None:
        out = server_proc.stdout.read().decode('utf-8', errors='ignore') if server_proc.stdout else ''
        raise RuntimeError(f'Server exited early. Output:\n{out}')
    raise RuntimeError('Server did not become healthy in time.')

print('Server is healthy at', ENV_URL)

In [ ]:
# Cell 4: Baseline (random + smart heuristic)
import json
from pathlib import Path

from scripts.run_baseline import RandomPolicy, SmartPolicy, evaluate_policy

baseline_random = evaluate_policy(ENV_URL, RandomPolicy(), n_episodes=EPISODES_EVAL, difficulty=DIFFICULTY)
baseline_smart = evaluate_policy(ENV_URL, SmartPolicy(), n_episodes=EPISODES_EVAL, difficulty=DIFFICULTY)

print('Random baseline:', baseline_random)
print('Smart heuristic:', baseline_smart)

Path('assets').mkdir(exist_ok=True)
Path('assets/baseline_results_judge.json').write_text(
    json.dumps({'random': baseline_random, 'smart': baseline_smart}, indent=2),
    encoding='utf-8',
)
print('Saved: assets/baseline_results_judge.json')

In [ ]:
# Cell 5: Real GRPO training using Unsloth-enabled script
# This runs against the environment-connected reward pipeline in train_grpo.py
import subprocess
import sys

cmd = [
    sys.executable,
    'train_grpo.py',
    '--model-id', MODEL_ID,
    '--difficulty', DIFFICULTY,
    '--dataset-size', str(DATASET_SIZE),
    '--num-epochs', str(NUM_EPOCHS),
    '--output-dir', OUTPUT_DIR,
    '--save-steps', '20',
    '--env-url', ENV_URL,
]

if JUDGE_READY:
    cmd.extend(['--judge-ready', '--min-update-steps', '80'])

if not USE_WANDB:
    # training still works without wandb account
    import os
    os.environ['WANDB_DISABLED'] = 'true'

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Training complete.')

In [ ]:
# Cell 6: Validate training depth evidence
import json
from pathlib import Path

summary_path = Path(OUTPUT_DIR) / 'training_summary.json'
assert summary_path.exists(), f'Missing {summary_path}'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2))

assert summary['planned_update_steps'] >= 80, 'Training too short for judge-grade evidence'
assert summary['reward_points'] >= 50, 'Too few reward points logged'
print('Training depth checks passed.')

In [ ]:
# Cell 7: Evaluate trained checkpoint vs random baseline (real model inference)
import subprocess
import sys

eval_cmd = [
    sys.executable,
    'scripts/eval_trained_vs_random.py',
    '--model-dir', OUTPUT_DIR,
    '--difficulty', DIFFICULTY,
    '--episodes', str(EPISODES_EVAL),
    '--seed', '42',
]
print('Running:', ' '.join(eval_cmd))
subprocess.run(eval_cmd, check=True)

update_cmd = [sys.executable, 'scripts/update_readme_evidence.py']
print('Running:', ' '.join(update_cmd))
subprocess.run(update_cmd, check=True)

print('Saved evidence: assets/trained_vs_random.json, assets/trained_vs_random.png')
print('README trained evidence block updated.')

In [ ]:
# Cell 8: Display key plots and metrics for judges
import json
from pathlib import Path
from IPython.display import Image, display

comparison = json.loads(Path('assets/trained_vs_random.json').read_text(encoding='utf-8'))
print(json.dumps(comparison, indent=2))

print('\nPlot: trained vs random')
display(Image(filename='assets/trained_vs_random.png'))

if Path('assets/reward_curves.png').exists():
    print('\nPlot: GRPO reward curves')
    display(Image(filename='assets/reward_curves.png'))

if Path('assets/loss_curve.png').exists():
    print('\nPlot: GRPO loss curve')
    display(Image(filename='assets/loss_curve.png'))

if Path('assets/component_rewards.png').exists():
    print('\nPlot: per-component reward curves')
    display(Image(filename='assets/component_rewards.png'))

## Judge Checklist (Notebook Output)

After running all cells, verify these artifacts exist in repo:

- `assets/baseline_results_judge.json`
- `assets/reward_curves.png`
- `assets/loss_curve.png`
- `assets/component_rewards.png`
- `assets/trained_vs_random.json`
- `assets/trained_vs_random.png`
- `assets/training_summary_latest.json`

README trained-evidence section is auto-updated by `scripts/update_readme_evidence.py` in Cell 7.

If wandb is enabled, add the exact run URL in README.